# Practica Agentes Inteligentes - Agente unificado con LangChain

Notebook completo de la practica. Toda la logica de los componentes (scraper de
SensaCine, cartelera de Madrid, conciertos via Wegow, agente de correos,
generador de calendarios) se expone como **tools de LangChain** y se orquesta
con un unico **AgentExecutor** sobre **ChatOllama (qwen2.5:1.5b)**.

Arquitectura:

```

  Usuario (Telegram | Alexa | Web | CLI | cron semanal)
       \
        \---->  AgentExecutor (LangChain) <----+
                       |                       |
                       v                       |
                ChatOllama qwen2.5:1.5b        |
                       |                       |
                       v   tool_calls          |
        +-------+-------+-------+-------+      |
        v       v       v       v       v      |
   movie_info cartelera concerts respond cal  ICS / Telegram
        |       |        |        |       |
        v       v        v        v       v
   SensaCine eCartelera Wegow LLM-coloc disco/Telegram
```

Apartados opcionales de la diapositiva 8 incluidos:
- Agente de conciertos semanal con filtro por artistas favoritos.
- Workflow N8N de guardarrail.
- Workflow ComfyUI con Ace Step para generacion de canciones.
- Agente de respuesta a correos.
- Agente de gestion de calendario (.ics).


## 1. Instalacion de dependencias

Incluye `langchain`, `langchain-ollama` y `langchain-community` ademas del resto.

In [1]:
# requirements.txt
%pip install -q \
    "requests>=2.31.0" \
    "beautifulsoup4>=4.12.0" \
    "lxml>=5.0.0" \
    "flask>=3.0.0" \
    "python-telegram-bot>=21.0" \
    "ask-sdk-core>=1.19.0" \
    "ollama>=0.4.0" \
    "langchain>=1.0.0" \
    "langchain-ollama>=1.0.0" \
    "langchain-community>=0.4.0"


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


## 2. Configuracion centralizada (`config.py`)

Unico lugar con tokens, URLs y nombres de modelo.

In [2]:
# config.py
import sys, types

config = types.ModuleType("config")

# --- Telegram ---
config.TELEGRAM_BOT_TOKEN = "8681004744:AAH-t0sPHD5Zr_2lMXpoKIYNXzE7n5U7YAY"
config.TELEGRAM_CHAT_ID = "6451572961"

# --- LLM (LangChain + Ollama) ---
config.OLLAMA_URL = "http://localhost:11434"
config.OLLAMA_MODEL = "qwen2.5:1.5b"
config.LLM_TEMPERATURE = 0.0

# --- Scraping ---
config.SENSACINE_BASE = "https://www.sensacine.com"
config.ECARTELERA_URL = "https://www.ecartelera.com"
config.WEGOW_API = "https://www.wegow.com/api/events?cities=3117735"  # 3117735 = Madrid
config.REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# --- Cache / persistencia ---
config.CACHE_FILE = "movie_cache.json"
config.USER_PROFILE_FILE = "user_profile.json"

# --- Web ---
config.FLASK_HOST = "0.0.0.0"
config.FLASK_PORT = 5000
config.FLASK_DEBUG = True

sys.modules["config"] = config
print(f"Modulo config registrado. LLM = {config.OLLAMA_MODEL}")


Modulo config registrado. LLM = qwen2.5:1.5b


## 3. Perfil de usuario (`user_profile.json`)

Generos con nota minima, directores favoritos y artistas favoritos para los conciertos.

In [3]:
# user_profile.json
import json

user_profile = {
    "genres": {
        "Sci-Fi": 6.0,
        "Action": 6.5,
        "Drama": 7.0,
        "Comedy": 6.0,
        "Horror": 5.5,
        "Animation": 7.0,
        "Thriller": 6.5,
        "Biography": 7.0
    },
    "favorite_directors": [
        "Christopher Nolan",
        "Denis Villeneuve",
        "Quentin Tarantino",
        "Pedro Almodovar",
        "Martin Scorsese"
    ],
    "favorite_artists": [
        "Eric Clapton",
        "Fito & Fitipaldis",
        "Arcangel",
        "Vetusta Morla",
        "Love of Lesbian",
        "Coldplay",
        "Radiohead"
    ]
}

with open("user_profile.json", "w", encoding="utf-8") as f:
    json.dump(user_profile, f, ensure_ascii=False, indent=2)

print("Perfil guardado en user_profile.json")
print(json.dumps(user_profile, ensure_ascii=False, indent=2))


Perfil guardado en user_profile.json
{
  "genres": {
    "Sci-Fi": 6.0,
    "Action": 6.5,
    "Drama": 7.0,
    "Comedy": 6.0,
    "Horror": 5.5,
    "Animation": 7.0,
    "Thriller": 6.5,
    "Biography": 7.0
  },
  "favorite_directors": [
    "Christopher Nolan",
    "Denis Villeneuve",
    "Quentin Tarantino",
    "Pedro Almodovar",
    "Martin Scorsese"
  ],
  "favorite_artists": [
    "Eric Clapton",
    "Fito & Fitipaldis",
    "Arcangel",
    "Vetusta Morla",
    "Love of Lesbian",
    "Coldplay",
    "Radiohead"
  ]
}


## 4. LLM unificado con LangChain (`ChatOllama`)

Toda la generacion de lenguaje (respuestas a correos, parrafadas del bot de
Telegram, razonamiento del agente) pasa por un mismo objeto `ChatOllama` para
que cualquier cambio de modelo se haga en **un solo sitio**.


In [4]:
from langchain_ollama import ChatOllama
import config

llm = ChatOllama(
    model=config.OLLAMA_MODEL,
    base_url=config.OLLAMA_URL,
    temperature=config.LLM_TEMPERATURE,
)

# Prueba minima
print(llm.invoke("Di hola en una sola frase").content)


¡Hola!


## 5. Tool 1: `movie_info` - Scraper de SensaCine

Web scraping con BeautifulSoup. Devuelve un diccionario con titulo, nota,
votos, director, sinopsis, duracion, genero, ano y poster (en base64).
Cachea resultados en `movie_cache.json` para no sobrecargar el servidor.


In [5]:
# movie_scraper.py
import base64, json, os, re, sys, types
import requests
from bs4 import BeautifulSoup
import config

SENSACINE_BASE = config.SENSACINE_BASE
SEARCH_URL = f"{SENSACINE_BASE}/busqueda/?q={{q}}"


def _load_cache():
    if os.path.exists(config.CACHE_FILE):
        try:
            with open(config.CACHE_FILE, encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}


def _save_cache(cache):
    with open(config.CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def _search_sensacine(title):
    url = SEARCH_URL.format(q=requests.utils.quote(title))
    r = requests.get(url, headers=config.REQUEST_HEADERS, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    a = soup.find("a", class_="meta-title-link")
    if not a or not a.get("href"):
        return None
    href = a["href"]
    if href.startswith("/"):
        href = SENSACINE_BASE + href
    return href


def _scrape_movie_page(url):
    r = requests.get(url, headers=config.REQUEST_HEADERS, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    title = (soup.find("div", class_="titlebar-title") or soup.find("h1"))
    title = title.get_text(strip=True) if title else ""

    nota_el = soup.find("span", class_="stareval-note")
    nota = nota_el.get_text(strip=True).replace(",", ".") if nota_el else "N/A"

    votos_el = soup.find("span", class_="stareval-review")
    votos = 0
    if votos_el:
        m = re.search(r"\d[\d.,]*", votos_el.get_text(" ", strip=True).replace(",", ""))
        if m:
            votos = int(m.group().replace(".", ""))

    director = "N/A"
    for li in soup.find_all("div", class_="meta-body-item"):
        if li.find("span", string=re.compile("Director", re.I)):
            link = li.find("a")
            if link:
                director = link.get_text(strip=True)
                break

    genre = ""
    for li in soup.find_all("div", class_="meta-body-info"):
        sp = li.find("span", string=re.compile("Genero|Generos", re.I))
        if sp:
            genres = [a.get_text(strip=True) for a in li.find_all("a")]
            genre = ", ".join(genres)
            break

    duracion = ""
    for li in soup.find_all("div", class_="meta-body-info"):
        m = re.search(r"\b\d+h\s*\d*m?\b|\b\d+\s*min\b", li.get_text(" "))
        if m:
            duracion = m.group()
            break

    year = ""
    yr = soup.find("span", class_="meta-body-info")
    if yr:
        m = re.search(r"\b(19|20)\d{2}\b", yr.get_text(" "))
        if m:
            year = m.group()

    sinopsis_el = soup.find("div", class_="content-txt")
    sinopsis = sinopsis_el.get_text(" ", strip=True) if sinopsis_el else ""

    poster_b64 = ""
    img = soup.find("img", class_="thumbnail-img")
    if img and img.get("src"):
        try:
            ir = requests.get(img["src"], headers=config.REQUEST_HEADERS, timeout=15)
            poster_b64 = base64.b64encode(ir.content).decode()[:500]
        except Exception:
            pass

    return {
        "titulo": title,
        "nota": nota,
        "nota_escala": "/5",
        "votos": votos,
        "director": director,
        "genero": genre,
        "duracion": duracion,
        "ano": year,
        "sinopsis": sinopsis,
        "url": url,
        "poster_b64": poster_b64,
    }


def get_movie_info(title, use_cache=True):
    cache = _load_cache()
    key = title.strip().lower()
    if use_cache and key in cache:
        return cache[key]
    url = _search_sensacine(title)
    if not url:
        return None
    info = _scrape_movie_page(url)
    cache[key] = info
    _save_cache(cache)
    return info


def format_movie_text(info):
    if not info:
        return "No encontrada."
    return (
        f"{info['titulo']} ({info.get('ano','?')})\n"
        f"  Nota: {info['nota']}{info['nota_escala']} ({info['votos']} votos)\n"
        f"  Director: {info['director']}\n"
        f"  Genero: {info['genero']}  Duracion: {info['duracion']}\n"
        f"  {info['sinopsis'][:300]}{'...' if len(info['sinopsis']) > 300 else ''}\n"
        f"  {info['url']}"
    )


# Registrar como modulo importable desde el resto del notebook
mod = types.ModuleType("movie_scraper")
mod.get_movie_info = get_movie_info
mod.format_movie_text = format_movie_text
sys.modules["movie_scraper"] = mod
print("movie_scraper cargado")


movie_scraper cargado


In [6]:
# Prueba rapida
info = get_movie_info("Inception")
print(format_movie_text(info) if info else "No encontrada")


Origen (?)
  Nota: 4.4/5 (4109 votos)
  Director: Christopher Nolan
  Genero: Ciencia ficción, Suspense  Duracion: 2h 28min
  Dom Cobb (Leonardo DiCaprio) es el mejor extractor. Su oficio consiste en introducirse en los sueños de sus víctimas y extraerle secretos del mundo de los negocios para luego venderlos con grandes dividendos. Debido a sus arriesgados métodos, grandes consorcios lo tienen en la mirilla, y ningún esco...
  https://www.sensacine.com/peliculas/pelicula-143692/


## 6. Tool 2: `cartelera_madrid` - Cartelera filtrada por perfil

Scrapea los principales cines de Madrid en eCartelera, deduplica por titulo,
enriquece con datos de SensaCine y aplica el filtro definido en
`user_profile.json` (genero+nota minima o director favorito).


In [7]:
# cartelera_scraper.py
import json, os, re, sys, time, types
import requests
from bs4 import BeautifulSoup
import config

CINES_MADRID = [
    ("Yelmo Cines Ideal",       "https://www.ecartelera.com/cines/54,0,1.html"),
    ("Callao",                  "https://www.ecartelera.com/cines/8,0,1.html"),
    ("Cinesa Proyecciones",     "https://www.ecartelera.com/cines/17,0,1.html"),
    ("Cines Princesa",          "https://www.ecartelera.com/cines/20,0,1.html"),
    ("Palacio de la Prensa",    "https://www.ecartelera.com/cines/38,0,1.html"),
    ("Renoir Plaza de Espana",  "https://www.ecartelera.com/cines/44,0,1.html"),
    ("Cinesa Principe Pio",     "https://www.ecartelera.com/cines/53,0,1.html"),
]


def _scrape_cinema(name, url):
    try:
        r = requests.get(url, headers=config.REQUEST_HEADERS, timeout=15)
        r.raise_for_status()
    except requests.RequestException:
        return []
    soup = BeautifulSoup(r.text, "lxml")
    out = []
    for item in soup.find_all("div", class_="titem"):
        title_el = item.find("p", class_="tit")
        if not title_el:
            continue
        link = title_el.find("a")
        title = (link.get_text(strip=True) if link else title_el.get_text(strip=True))
        url_film = link.get("href", "") if link else ""
        data = item.find("p", class_="data")
        spans = data.find_all("span") if data else []
        out.append({
            "titulo": title,
            "url_ecartelera": url_film,
            "duracion": spans[0].get_text(strip=True) if len(spans) > 0 else "",
            "pais":     spans[1].get_text(strip=True) if len(spans) > 1 else "",
            "genero":   spans[2].get_text(strip=True) if len(spans) > 2 else "",
            "cine":     name,
        })
    return out


def get_cartelera_madrid():
    seen, out = set(), []
    for name, url in CINES_MADRID:
        for m in _scrape_cinema(name, url):
            key = m["titulo"].lower().strip()
            if key in seen:
                continue
            seen.add(key)
            out.append(m)
        time.sleep(0.3)
    return out


def enrich_with_sensacine(movies):
    from movie_scraper import get_movie_info
    for m in movies:
        info = get_movie_info(m["titulo"])
        if info:
            m["nota_sensacine"] = info["nota"]
            m["director"]       = info.get("director", m.get("director", ""))
            m["sinopsis"]       = info.get("sinopsis", "")
            m["genero_sc"]      = info.get("genero", "")
            m["url_sensacine"]  = info.get("url", "")
        else:
            m["nota_sensacine"] = "N/A"
    return movies


def load_user_profile():
    if os.path.exists(config.USER_PROFILE_FILE):
        with open(config.USER_PROFILE_FILE, encoding="utf-8") as f:
            return json.load(f)
    return {"genres": {}, "favorite_directors": []}


_SC_TO_PROFILE = {
    "Ciencia ficcion": "Sci-Fi",
    "Ciencia ficcion": "Sci-Fi",
    "Accion": "Action",
    "Drama": "Drama",
    "Comedia": "Comedy",
    "Terror": "Horror",
    "Animacion": "Animation",
    "Animacion": "Animation",
    "Thriller": "Thriller",
    "Biografia": "Biography",
}


def filter_by_profile(movies, profile=None):
    if profile is None:
        profile = load_user_profile()
    favs = [d.lower() for d in profile.get("favorite_directors", [])]
    rules = profile.get("genres", {})
    out = []
    for m in movies:
        director = (m.get("director") or "").lower()
        if any(f in director for f in favs):
            m["_filter_reason"] = f"director favorito"
            out.append(m)
            continue
        nota = m.get("nota_sensacine")
        try:
            nota_v = float(nota) if nota and nota != "N/A" else None
        except ValueError:
            nota_v = None
        gen_sc = m.get("genero_sc") or m.get("genero", "")
        for sc_g, profile_g in _SC_TO_PROFILE.items():
            if sc_g.lower() in gen_sc.lower():
                threshold = rules.get(profile_g)
                if threshold and nota_v is not None and nota_v >= threshold:
                    m["_filter_reason"] = f"{profile_g} >= {threshold}"
                    out.append(m)
                    break
    return out


def format_cartelera_text(movies):
    if not movies:
        return "Sin peliculas que pasen el filtro."
    lines = ["CARTELERA FILTRADA - MADRID", "=" * 30, ""]
    for m in movies:
        lines.append(f"- {m['titulo']}  [{m.get('_filter_reason','')}]")
        lines.append(f"   Cine: {m.get('cine','?')}  | Nota SC: {m.get('nota_sensacine','?')}")
        if m.get("director"):
            lines.append(f"   Director: {m['director']}")
        lines.append("")
    return "\n".join(lines)


mod = types.ModuleType("cartelera_scraper")
mod.get_cartelera_madrid = get_cartelera_madrid
mod.enrich_with_sensacine = enrich_with_sensacine
mod.filter_by_profile = filter_by_profile
mod.format_cartelera_text = format_cartelera_text
mod.load_user_profile = load_user_profile
sys.modules["cartelera_scraper"] = mod
print("cartelera_scraper cargado")


cartelera_scraper cargado


## 7. Tool 3: `conciertos_semana` - Conciertos en Madrid + filtro por artistas

Consulta la API JSON publica de Wegow para Madrid (`cities=3117735`), filtra
por los proximos 7 dias y opcionalmente solo deja pasar los artistas favoritos
del perfil. Implementa el opcional 1 de la diapositiva 8.


In [8]:
# concerts_scraper.py
import json, sys, types
from datetime import datetime, timedelta, timezone
import requests
import config


def fetch_concerts(limit_days=7):
    try:
        r = requests.get(config.WEGOW_API, headers=config.REQUEST_HEADERS, timeout=20)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"Error wegow: {e}", file=sys.stderr)
        return []
    data = r.json()
    events = data.get("events", []) if isinstance(data, dict) else []
    today = datetime.now(timezone.utc).date()
    deadline = today + timedelta(days=limit_days)
    out = []
    for e in events:
        city = e.get("city") or {}
        if city.get("name") != "Madrid":
            continue
        sd = e.get("start_date")
        if not sd:
            continue
        try:
            dt = datetime.strptime(sd[:10], "%Y-%m-%d").date()
        except ValueError:
            continue
        if not (today <= dt <= deadline):
            continue
        venue = e.get("venue") or {}
        out.append({
            "id": e.get("id"),
            "titulo": e.get("title"),
            "fecha": dt.isoformat(),
            "hora":  sd[11:16] if len(sd) >= 16 else "",
            "artistas": [a.get("name") for a in (e.get("artists") or [])],
            "recinto": venue.get("name") or "",
            "url": e.get("permalink") or e.get("purchase_url") or "",
        })
    out.sort(key=lambda c: c["fecha"])
    return out


def filter_by_favorite_artists(concerts, profile=None):
    if profile is None:
        with open("user_profile.json", encoding="utf-8") as f:
            profile = json.load(f)
    favs = [a.lower() for a in profile.get("favorite_artists", [])]
    if not favs:
        return concerts
    out = []
    for c in concerts:
        for a in c["artistas"]:
            if a and a.lower() in favs:
                c["_match"] = a
                out.append(c)
                break
    return out


def format_concerts_text(concerts, only_favorites=False):
    if not concerts:
        return ("Sin conciertos favoritos esta semana." if only_favorites
                else "Sin conciertos esta semana.")
    title = ("CONCIERTOS - ARTISTAS FAVORITOS" if only_favorites
             else "CONCIERTOS EN MADRID (7 DIAS)")
    lines = [title, "=" * len(title), ""]
    for c in concerts:
        lines.append(f"[{c['fecha']} {c['hora']}] {c['titulo']}")
        if c["artistas"]:
            lines.append(f"   Artistas: {', '.join(c['artistas'][:5])}")
        if c["recinto"]:
            lines.append(f"   Sala: {c['recinto']}")
        if c.get("_match"):
            lines.append(f"   *** Favorito: {c['_match']}")
        if c["url"]:
            lines.append(f"   {c['url']}")
        lines.append("")
    return "\n".join(lines)


mod = types.ModuleType("concerts_scraper")
mod.fetch_concerts = fetch_concerts
mod.filter_by_favorite_artists = filter_by_favorite_artists
mod.format_concerts_text = format_concerts_text
sys.modules["concerts_scraper"] = mod
print("concerts_scraper cargado")


concerts_scraper cargado


## 8. Tool 4: `responder_correo` - Atencion al cliente con sentimiento

Detecta sentimiento (favorable / desfavorable / neutral) con un clasificador
lexico ligero y delega la generacion de la respuesta al **mismo `ChatOllama`**
del nucleo (no abre una conexion paralela a Ollama). Implementa el ejemplo de
la diapositiva 9.


In [9]:
# email_agent.py
import re, sys, types
from langchain_core.messages import SystemMessage, HumanMessage

POSITIVE_WORDS = {
    "gracias","genial","excelente","fantastico","increible","perfecto",
    "maravilloso","recomiendo","feliz","contento","alegra","alegro",
    "agradable","disfrute","disfrutado","encanta","encantado","buen","buena",
}
NEGATIVE_WORDS = {
    "frio","fria","horrible","malo","mala","pesimo","asco","sucio","sucia",
    "caro","cara","lento","lenta","tarde","roto","rota","reclamo","queja",
    "defectuoso","defectuosa","problema","esperar","decepcion",
}


def classify_sentiment(text):
    norm = text.lower()
    pos = sum(1 for w in POSITIVE_WORDS if w in norm)
    neg = sum(1 for w in NEGATIVE_WORDS if w in norm)
    if pos > neg:  return "favorable", pos, neg
    if neg > pos:  return "desfavorable", pos, neg
    return "neutral", pos, neg


EMAIL_SYSTEM_PROMPT = (
    "Eres un agente de atencion al cliente cordial. "
    "Responde al mensaje del cliente con un tono adecuado a su sentimiento. "
    "Maximo 3 frases. "
    "- Si es desfavorable: pide disculpas y ofrece una accion correctiva. "
    "- Si es favorable: agradece sinceramente y refuerza el vinculo. "
    "- Si es neutral: contesta con informacion util."
)


def email_agent(text, llm_obj=None):
    sentiment, pos, neg = classify_sentiment(text)
    llm_local = llm_obj or llm
    msg = llm_local.invoke([
        SystemMessage(content=EMAIL_SYSTEM_PROMPT),
        HumanMessage(content=f"[sentimiento detectado: {sentiment}]\nMensaje: {text}"),
    ])
    return {
        "sentimiento": sentiment,
        "score": (pos, neg),
        "respuesta": msg.content.strip(),
    }


mod = types.ModuleType("email_agent")
mod.classify_sentiment = classify_sentiment
mod.email_agent = email_agent
sys.modules["email_agent"] = mod
print("email_agent cargado")


email_agent cargado


## 9. Tool 5: `generar_calendario` - Exportar a `.ics`

Convierte conciertos o cartelera en un calendario `.ics` (RFC 5545) importable
en Google Calendar, Outlook o Apple Calendar. Implementa el opcional de
gestion de calendario.


In [10]:
# calendar_agent.py
import hashlib, os, sys, types
from datetime import datetime, timedelta, time, timezone

ICS_HEADER = "BEGIN:VCALENDAR\r\nVERSION:2.0\r\nPRODID:-//Agente Peliculas//ES\r\nCALSCALE:GREGORIAN\r\n"
ICS_FOOTER = "END:VCALENDAR\r\n"


def _esc(t):
    if not t:
        return ""
    return t.replace("\\","\\\\").replace(",","\\,").replace(";","\\;").replace("\n","\\n")


def _event(uid, dtstart, dtend, summary, description, location, url=None):
    fmt = "%Y%m%dT%H%M%S"
    fields = [
        "BEGIN:VEVENT",
        f"UID:{uid}@agente-peliculas",
        f"DTSTAMP:{datetime.now(timezone.utc).strftime(fmt)}Z",
        f"DTSTART:{dtstart.strftime(fmt)}",
        f"DTEND:{dtend.strftime(fmt)}",
        f"SUMMARY:{_esc(summary)}",
        f"DESCRIPTION:{_esc(description)}",
        f"LOCATION:{_esc(location)}",
    ]
    if url:
        fields.append(f"URL:{_esc(url)}")
    fields.append("END:VEVENT")
    return "\r\n".join(fields) + "\r\n"


def concerts_to_ics(concerts, output_path="agenda_conciertos.ics"):
    body = ICS_HEADER
    for c in concerts:
        try:
            ymd = c["fecha"]
            hh, mm = (c.get("hora") or "21:00").split(":")[:2]
            dt = datetime.strptime(f"{ymd} {hh}:{mm}", "%Y-%m-%d %H:%M")
        except Exception:
            dt = datetime.combine(datetime.strptime(c["fecha"], "%Y-%m-%d").date(), time(21, 0))
        end = dt + timedelta(hours=2)
        artists = ", ".join(c.get("artistas", []))
        uid = hashlib.md5(f"{c.get('id','')}-{c.get('titulo','')}".encode()).hexdigest()
        body += _event(uid, dt, end, c.get("titulo", "Concierto"),
                       f"Artistas: {artists}", c.get("recinto", ""), c.get("url"))
    body += ICS_FOOTER
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(body)
    return os.path.abspath(output_path)


def cartelera_to_ics(movies, output_path="agenda_cartelera.ics"):
    body = ICS_HEADER
    for m in movies:
        title = m.get("titulo", "Pelicula")
        dt = datetime.combine(datetime.now().date(), time(21, 0))
        uid = hashlib.md5(f"{title}-{m.get('cine','')}".encode()).hexdigest()
        body += _event(uid, dt, dt + timedelta(hours=2), f"Cine: {title}",
                       f"Director: {m.get('director','')}\nNota: {m.get('nota_sensacine','')}\n{m.get('sinopsis','')[:200]}",
                       m.get("cine", ""), m.get("url_sensacine"))
    body += ICS_FOOTER
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(body)
    return os.path.abspath(output_path)


mod = types.ModuleType("calendar_agent")
mod.concerts_to_ics = concerts_to_ics
mod.cartelera_to_ics = cartelera_to_ics
sys.modules["calendar_agent"] = mod
print("calendar_agent cargado")


calendar_agent cargado


## 10. Agente unificado con LangChain

Cada modulo anterior se envuelve en un `@tool`. El `AgentExecutor` con
`create_tool_calling_agent` decide en cada turno cual invocar segun el
mensaje en lenguaje natural.

| Tool | Descripcion |
|------|-------------|
| `movie_info` | Devuelve nota, director, sinopsis... de una pelicula |
| `cartelera_madrid` | Cartelera de Madrid filtrada por perfil |
| `conciertos_semana` | Conciertos en Madrid los proximos 7 dias |
| `responder_correo` | Respuesta a correo de cliente con sentimiento |
| `generar_calendario` | Exporta cartelera o conciertos a `.ics` |


In [11]:
from langchain_core.tools import tool
from langchain.agents import create_agent

from movie_scraper import get_movie_info, format_movie_text
from cartelera_scraper import (
    get_cartelera_madrid, enrich_with_sensacine, filter_by_profile, format_cartelera_text,
)
from concerts_scraper import (
    fetch_concerts, filter_by_favorite_artists, format_concerts_text,
)
from email_agent import email_agent
from calendar_agent import concerts_to_ics, cartelera_to_ics


@tool
def movie_info(title: str) -> str:
    """Devuelve nota, votos, director, genero, duracion y sinopsis de una pelicula
    a partir de su titulo. Usa SensaCine como fuente."""
    info = get_movie_info(title)
    return format_movie_text(info) if info else f"No se encontro la pelicula: {title}"


@tool
def cartelera_madrid() -> str:
    """Devuelve la cartelera actual de Madrid filtrada por el perfil del usuario."""
    movies = get_cartelera_madrid()
    movies = enrich_with_sensacine(movies)
    movies = filter_by_profile(movies)
    return format_cartelera_text(movies[:10])


@tool
def conciertos_semana(only_favoritos: bool = False) -> str:
    """Devuelve los conciertos en Madrid de los proximos 7 dias.
    Si only_favoritos=True filtra por los artistas favoritos del perfil."""
    cs = fetch_concerts(limit_days=7)
    if only_favoritos:
        cs = filter_by_favorite_artists(cs)
    return format_concerts_text(cs, only_favorites=only_favoritos)


@tool
def responder_correo(mensaje: str) -> str:
    """Analiza el sentimiento de un correo de cliente y genera una respuesta
    contextualizada (favorable / desfavorable / neutral)."""
    out = email_agent(mensaje, llm_obj=llm)
    return f"[{out['sentimiento']}] {out['respuesta']}"


@tool
def generar_calendario(tipo: str = "conciertos") -> str:
    """Genera un fichero .ics importable en Google Calendar.
    tipo='conciertos' o tipo='cartelera'."""
    if tipo == "cartelera":
        movies = get_cartelera_madrid()
        movies = enrich_with_sensacine(movies)
        movies = filter_by_profile(movies)
        path = cartelera_to_ics(movies[:10])
    else:
        cs = fetch_concerts(limit_days=7)
        path = concerts_to_ics(cs)
    return f"ICS generado en: {path}"


TOOLS = [movie_info, cartelera_madrid, conciertos_semana, responder_correo, generar_calendario]

SYSTEM_PROMPT = """Eres un agente que SIEMPRE usa herramientas (tools) antes de responder.

REGLAS OBLIGATORIAS:
1. Si el usuario menciona el nombre de una pelicula y pide cualquier dato (nota, director, sinopsis, votos, duracion, genero, ano), llama INMEDIATAMENTE a movie_info(title=<nombre>) sin pedir aclaraciones.
2. Si pide la cartelera, llama a cartelera_madrid().
3. Si pide conciertos:
   - si menciona 'mis favoritos', 'favoritos' o 'mis artistas', llama a conciertos_semana(only_favoritos=True).
   - en otro caso, conciertos_semana(only_favoritos=False).
4. Si pide responder a un correo o mensaje de cliente, llama a responder_correo(mensaje=<texto del cliente>).
5. Si pide exportar al calendario o generar un .ics, llama a generar_calendario(tipo='conciertos') o tipo='cartelera' segun corresponda.
6. NUNCA inventes datos. NUNCA pidas aclaraciones si la query basta para llamar a una tool.
7. Despues de recibir el resultado de la tool, devuelve ese resultado al usuario en espanol; resume si es muy largo.
"""

agent = create_agent(llm, TOOLS, system_prompt=SYSTEM_PROMPT)


def ask(text: str) -> str:
    """Helper para hacer una query al agente y devolver solo el texto final."""
    out = agent.invoke({"messages": [("user", text)]})
    return out["messages"][-1].content


print(f"Agente listo con {len(TOOLS)} tools: {[t.name for t in TOOLS]}")


Agente listo con 5 tools: ['movie_info', 'cartelera_madrid', 'conciertos_semana', 'responder_correo', 'generar_calendario']


## 11. Demo: queries en lenguaje natural al agente unificado

In [12]:
queries = [
    "Cual es la nota de Inception y quien la dirige?",
    "Dame los conciertos favoritos de esta semana en Madrid",
    "Responde a este correo: 'La comida estaba fria y tarde'",
]

for q in queries:
    print("=" * 70)
    print("USER:", q)
    print("AGENT:", ask(q))
    print()


USER: Cual es la nota de Inception y quien la dirige?


AGENT: La nota de Inception es 4.4/5 con 4109 votos. La película está dirigida por Christopher Nolan y se trata de una cinta de ciencia ficción y suspense. Durante la película, Dom Cobb (interpretado por Leonardo DiCaprio) es un experto en extraer información de los sueños de sus víctimas para venderlas a grandes consorcios.

USER: Dame los conciertos favoritos de esta semana en Madrid


AGENT: ¡Hola! Aquí tienes los conciertos favoritos de esta semana en Madrid:

1. **Concierto de Eric Clapton** el 7 de mayo a las 21:00 horas.
   - Sala: Movistar Arena
   - Artistas: Eric Clapton

2. **Concierto de Fito & Fitipaldis** los días 8 y 9 de mayo a las 20:30 horas.
   - Sala: Movistar Arena
   - Artistas: Fito & Fitipaldis

¡No te pierdas estos conciertos!

USER: Responde a este correo: 'La comida estaba fria y tarde'


AGENT: Lo siento por la inconveniencia, pero no puedo responder a ese correo.



## 12. Frontends que delegan en el agente

Cada interfaz (Telegram, Alexa, Web, CLI) reduce su trabajo a:
1. Recoger el texto del usuario.
2. Llamar a `executor.invoke({"input": texto})`.
3. Devolver la respuesta.

Asi cualquier mejora del LLM o de los tools se propaga automaticamente a las
4 interfaces.


### 12.1 CLI

In [13]:
# Uso por linea de comandos: python movie_scraper.py "Inception"
# o consulta libre al agente:
def cli_query(text):
    return ask(text)

print(cli_query("dame la nota de Pulp Fiction"))


La nota de Pulp Fiction es 4.5/5 con 4011 votos. La película está dirigida por Quentin Tarantino y se trata de un género de Crimen, Drama. Tiene una duración de 2h 29min.


### 12.2 Bot de Telegram (`telegram_bot.py`)

Delega cualquier mensaje al `AgentExecutor`. Mantiene comandos rapidos como `/cartelera` y `/conciertos` para uso directo.

In [14]:
# telegram_bot.py
import sys, types

try:
    from telegram import Update
    from telegram.ext import Application, CommandHandler, MessageHandler, ContextTypes, filters
    TELEGRAM_AVAILABLE = True
except ImportError:
    TELEGRAM_AVAILABLE = False

import config


if TELEGRAM_AVAILABLE:
    async def start(update, context):
        await update.message.reply_text(
            "Agente de Peliculas y Conciertos. Escribe lo que quieras saber:\n"
            "ejemplos:\n"
            "  - 'que nota tiene Interstellar?'\n"
            "  - '/cartelera'\n"
            "  - '/conciertos favoritos'\n"
            "  - 'responde a: la pizza estaba fria'"
        )

    async def cartelera_cmd(update, context):
        out = ask("Dame la cartelera filtrada por mi perfil")
        await update.message.reply_text(out[:4000])

    async def conciertos_cmd(update, context):
        only = "favoritos" in " ".join(context.args).lower() if context.args else False
        out = ask("Dame mis conciertos favoritos esta semana" if only
                  else "Que conciertos hay esta semana en Madrid")
        await update.message.reply_text(out[:4000])

    async def free_text(update, context):
        out = ask(update.message.text)
        await update.message.reply_text(out[:4000])

    def build_app():
        app = Application.builder().token(config.TELEGRAM_BOT_TOKEN).build()
        app.add_handler(CommandHandler("start", start))
        app.add_handler(CommandHandler("cartelera", cartelera_cmd))
        app.add_handler(CommandHandler("conciertos", conciertos_cmd))
        app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, free_text))
        return app

    print("telegram_bot listo. Para arrancar: build_app().run_polling()")
else:
    print("(python-telegram-bot no instalado, pero el codigo queda definido)")


def send_telegram(message, chat_id=None):
    """Envio puntual sin levantar el bot."""
    import requests
    token = config.TELEGRAM_BOT_TOKEN
    chat_id = chat_id or config.TELEGRAM_CHAT_ID
    if not token or not chat_id:
        return False
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    for i in range(0, len(message), 4000):
        try:
            requests.post(url, data={"chat_id": chat_id, "text": message[i:i+4000], "parse_mode": "HTML"}, timeout=15)
        except Exception:
            return False
    return True


print("send_telegram disponible")


(python-telegram-bot no instalado, pero el codigo queda definido)
send_telegram disponible


### 12.3 Skill de Alexa (`alexa_lambda.py`)

Lambda Python que enruta cada intent al `AgentExecutor` (envuelto detras de un import perezoso porque AWS Lambda no tendria Ollama: en produccion bastaria con servir el modelo en una EC2 o reemplazar `ChatOllama` por `ChatBedrock`).

In [15]:
# alexa_lambda.py
import sys, types

try:
    from ask_sdk_core.skill_builder import SkillBuilder
    from ask_sdk_core.dispatch_components import AbstractRequestHandler
    from ask_sdk_core.utils import is_intent_name, is_request_type
    from ask_sdk_core.handler_input import HandlerInput
    ASK_AVAILABLE = True
except ImportError:
    ASK_AVAILABLE = False


if ASK_AVAILABLE:
    class LaunchHandler(AbstractRequestHandler):
        def can_handle(self, h): return is_request_type("LaunchRequest")(h)
        def handle(self, h):
            return h.response_builder.speak(
                "Bienvenido al agente de peliculas y conciertos. "
                "Pregunta por una pelicula o por los conciertos de la semana."
            ).response

    class GetMovieIntent(AbstractRequestHandler):
        """Pregunta por cualquier dato de una pelicula. Slot `Movie`."""
        def can_handle(self, h): return is_intent_name("GetMovieIntent")(h)
        def handle(self, h):
            slots = h.request_envelope.request.intent.slots
            title = slots["Movie"].value if "Movie" in slots else ""
            out = ask(f"Dame la informacion de {title}")
            return h.response_builder.speak(out[:600]).response

    class ConcertsIntent(AbstractRequestHandler):
        def can_handle(self, h): return is_intent_name("ConcertsIntent")(h)
        def handle(self, h):
            out = ask("Que conciertos favoritos hay esta semana")
            return h.response_builder.speak(out[:600]).response

    sb = SkillBuilder()
    sb.add_request_handler(LaunchHandler())
    sb.add_request_handler(GetMovieIntent())
    sb.add_request_handler(ConcertsIntent())
    handler = sb.lambda_handler()
    print("alexa_lambda cargado. handler =", handler)
else:
    print("(ask-sdk-core no instalado)")


(ask-sdk-core no instalado)


### 12.4 Modelo de interaccion de Alexa (`alexa_interaction_model.json`)

In [16]:
import json

interaction_model = {
    "interactionModel": {
        "languageModel": {
            "invocationName": "agente de peliculas",
            "intents": [
                {"name": "AMAZON.CancelIntent", "samples": []},
                {"name": "AMAZON.StopIntent",   "samples": []},
                {"name": "AMAZON.HelpIntent",   "samples": []},
                {"name": "GetMovieIntent",
                 "slots": [{"name": "Movie", "type": "AMAZON.SearchQuery"}],
                 "samples": [
                     "dame informacion de {Movie}",
                     "que tal es {Movie}",
                     "cual es la nota de {Movie}",
                     "quien dirige {Movie}",
                 ]},
                {"name": "ConcertsIntent",
                 "samples": ["que conciertos hay esta semana", "dame los conciertos favoritos"]},
            ],
        }
    }
}

with open("alexa_interaction_model.json", "w", encoding="utf-8") as f:
    json.dump(interaction_model, f, ensure_ascii=False, indent=2)
print("alexa_interaction_model.json escrito")


alexa_interaction_model.json escrito


### 12.5 Web Flask (`web_app.py`)

Formulario web -> llamada al agente -> render del resultado.

In [17]:
# web_app.py
try:
    from flask import Flask, request, jsonify
    FLASK_AVAILABLE = True
except ImportError:
    FLASK_AVAILABLE = False

import config

if FLASK_AVAILABLE:
    app = Flask(__name__)

    INDEX_HTML = """
    <!doctype html>
    <html><head><meta charset="utf-8"><title>Agente de Peliculas</title></head>
    <body style="font-family: system-ui; max-width: 700px; margin: 2em auto;">
      <h1>Agente de Peliculas y Conciertos</h1>
      <form method="post" action="/q">
        <input name="q" style="width:80%" placeholder="cual es la nota de Inception?" />
        <button>Preguntar</button>
      </form>
      <pre>{{out}}</pre>
    </body></html>
    """

    @app.route("/", methods=["GET"])
    def index():
        return INDEX_HTML.replace("{{out}}", "")

    @app.route("/q", methods=["POST"])
    def query():
        text = request.form.get("q", "").strip()
        if not text:
            return INDEX_HTML.replace("{{out}}", "")
        out = ask(text)
        return INDEX_HTML.replace("{{out}}", out)

    @app.route("/api/q", methods=["POST"])
    def api_query():
        text = request.json.get("q", "")
        out = ask(text)
        return jsonify({"output": out})

    print("web_app listo. Para arrancar: app.run(host=config.FLASK_HOST, port=config.FLASK_PORT)")
else:
    print("(Flask no instalado)")


web_app listo. Para arrancar: app.run(host=config.FLASK_HOST, port=config.FLASK_PORT)


## 13. Workflows externos opcionales (slide 8)

### 13.1 N8N - Guardarrail sobre el agente

Webhook -> validacion entrada -> LLM -> validacion salida -> respuesta. JSON exportable importable directamente en cualquier instancia de N8N.

In [18]:
import json

n8n_guardrail_workflow = {
    "name": "LLM Guardrail",
    "nodes": [
        {"parameters":{"httpMethod":"POST","path":"llm-guardrail","options":{}},
         "id":"1","name":"Webhook In","type":"n8n-nodes-base.webhook","typeVersion":1,"position":[240,300]},
        {"parameters":{"jsCode":(
            "const prompt = ($input.first().json.body && $input.first().json.body.prompt) || '';\n"
            "const blocked = [/ignore (all|previous) instructions/i,/system prompt/i,/jailbreak/i,/\\b\\d{16}\\b/];\n"
            "for (const re of blocked) { if (re.test(prompt)) return [{json:{allowed:false,reason:'guardrail entrada',prompt}}]; }\n"
            "return [{json:{allowed:true,prompt}}];")},
         "id":"2","name":"Validate Input","type":"n8n-nodes-base.code","typeVersion":2,"position":[460,300]},
        {"parameters":{"conditions":{"boolean":[{"value1":"={{$json.allowed}}","value2":True}]}},
         "id":"3","name":"If Input Allowed","type":"n8n-nodes-base.if","typeVersion":1,"position":[680,300]},
        {"parameters":{"resource":"chat","model":"gpt-4o-mini",
                       "messages":{"values":[
                           {"role":"system","content":"Eres un asistente experto."},
                           {"role":"user","content":"={{$json.prompt}}"}]}},
         "id":"4","name":"OpenAI LLM","type":"n8n-nodes-base.openAi","typeVersion":1,"position":[900,200]},
        {"parameters":{"jsCode":(
            "const out = ($input.first().json.message && $input.first().json.message.content) || '';\n"
            "const banned = [/idiota/i,/imbecil/i,/matar/i,/suicid/i];\n"
            "for (const re of banned) { if (re.test(out)) return [{json:{safe:false,sanitized:'[bloqueado salida]'}}]; }\n"
            "return [{json:{safe:true,sanitized:out}}];")},
         "id":"5","name":"Validate Output","type":"n8n-nodes-base.code","typeVersion":2,"position":[1120,200]},
        {"parameters":{"respondWith":"json","responseBody":"={{ {ok: $json.safe, response: $json.sanitized} }}"},
         "id":"6","name":"Respond Ok","type":"n8n-nodes-base.respondToWebhook","typeVersion":1,"position":[1340,200]},
        {"parameters":{"respondWith":"json","responseBody":"={{ {ok: false, reason: $json.reason} }}"},
         "id":"7","name":"Respond Blocked","type":"n8n-nodes-base.respondToWebhook","typeVersion":1,"position":[900,420]},
    ],
    "connections": {
        "Webhook In":      {"main":[[{"node":"Validate Input","type":"main","index":0}]]},
        "Validate Input":  {"main":[[{"node":"If Input Allowed","type":"main","index":0}]]},
        "If Input Allowed":{"main":[
            [{"node":"OpenAI LLM","type":"main","index":0}],
            [{"node":"Respond Blocked","type":"main","index":0}]]},
        "OpenAI LLM":      {"main":[[{"node":"Validate Output","type":"main","index":0}]]},
        "Validate Output": {"main":[[{"node":"Respond Ok","type":"main","index":0}]]},
    },
    "active": False, "settings": {}, "id": "llm-guardrail",
}

with open("n8n_guardrail_workflow.json", "w", encoding="utf-8") as f:
    json.dump(n8n_guardrail_workflow, f, ensure_ascii=False, indent=2)
print("n8n_guardrail_workflow.json escrito  (nodos:", len(n8n_guardrail_workflow["nodes"]), ")")


n8n_guardrail_workflow.json escrito  (nodos: 7 )


### 13.2 ComfyUI - Workflow de Ace Step para generacion de canciones

In [19]:
comfyui_acestep_workflow = {
    "3": {"class_type":"CheckpointLoaderSimple","inputs":{"ckpt_name":"ace_step_v1.safetensors"}},
    "4": {"class_type":"EmptyAceStepLatentAudio","inputs":{"seconds":30,"batch_size":1}},
    "5": {"class_type":"TextEncodeAceStepAudio","inputs":{
        "clip":["3",1],
        "tags":"electronic, indie pop, 110 bpm, female vocal, dreamy, lo-fi",
        "lyrics":"[verse]\nWalking through the city lights tonight\nEvery corner tells a different story\n[chorus]\nWe are the ones who never sleep\nChasing dreams across the streets\n",
        "lyrics_strength":1.0}},
    "6": {"class_type":"TextEncodeAceStepAudio","inputs":{"clip":["3",1],"tags":"silence","lyrics":"","lyrics_strength":1.0}},
    "7": {"class_type":"KSampler","inputs":{"model":["3",0],"positive":["5",0],"negative":["6",0],
        "latent_image":["4",0],"seed":42,"steps":50,"cfg":5.0,"sampler_name":"euler","scheduler":"simple","denoise":1.0}},
    "8": {"class_type":"VAEDecodeAudio","inputs":{"samples":["7",0],"vae":["3",2]}},
    "9": {"class_type":"SaveAudio","inputs":{"audio":["8",0],"filename_prefix":"ace_step_song"}},
}

with open("comfyui_acestep_workflow.json", "w", encoding="utf-8") as f:
    json.dump(comfyui_acestep_workflow, f, ensure_ascii=False, indent=2)
print("comfyui_acestep_workflow.json escrito (nodos:", len(comfyui_acestep_workflow), ")")


comfyui_acestep_workflow.json escrito (nodos: 7 )


## 14. Automatizacion: cron semanal

Lunes a las 9:00, el script invoca al agente con dos queries y envia el
resultado por Telegram. Asi cubre los apartados:
- cartelera filtrada por perfil (slide 3),
- conciertos de la semana filtrados por artistas favoritos (slide 8).


In [20]:
# Generamos un .py independiente y un .sh que cron invoca.
import os, json

CONCERTS_PY_LINES = [
    "#!/usr/bin/env python3",
    "import json, os, sys",
    "import requests",
    "from datetime import datetime, timedelta, timezone",
    "",
    "sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))",
    "import config",
    "",
    'WEGOW = "https://www.wegow.com/api/events?cities=3117735"',
    "r = requests.get(WEGOW, headers=config.REQUEST_HEADERS, timeout=20)",
    'events = r.json().get("events", [])',
    "today = datetime.now(timezone.utc).date()",
    "deadline = today + timedelta(days=7)",
    "",
    'with open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "user_profile.json"), encoding="utf-8") as f:',
    "    profile = json.load(f)",
    'favs = [a.lower() for a in profile.get("favorite_artists", [])]',
    "",
    "picked = []",
    "for e in events:",
    '    city = e.get("city") or {}',
    '    if city.get("name") != "Madrid":',
    "        continue",
    '    sd = e.get("start_date") or ""',
    "    if not sd:",
    "        continue",
    "    try:",
    '        d = datetime.strptime(sd[:10], "%Y-%m-%d").date()',
    "    except ValueError:",
    "        continue",
    "    if not (today <= d <= deadline):",
    "        continue",
    '    artists = [a.get("name") for a in (e.get("artists") or [])]',
    "    for a in artists:",
    "        if a and a.lower() in favs:",
    "            picked.append({",
    '                "fecha": d.isoformat(),',
    '                "hora":  sd[11:16] if len(sd) >= 16 else "",',
    '                "titulo": e.get("title", ""),',
    '                "recinto": (e.get("venue") or {}).get("name", ""),',
    '                "match":  a,',
    '                "url":    e.get("permalink", ""),',
    "            })",
    "            break",
    "",
    "if not picked:",
    '    print("Sin conciertos favoritos esta semana.")',
    "    sys.exit(0)",
    "",
    'parts = ["<b>Conciertos favoritos esta semana</b>", ""]',
    "for c in picked:",
    "    parts.append(c['fecha'] + ' ' + c['hora'])",
    "    parts.append(c['titulo'])",
    "    parts.append('Sala: ' + c['recinto'])",
    "    parts.append('Artista favorito: ' + c['match'])",
    "    parts.append('<a href=\"' + c['url'] + '\">Mas info</a>')",
    "    parts.append('')",
    "msg = chr(10).join(parts)",
    "",
    "token   = config.TELEGRAM_BOT_TOKEN",
    "chat_id = config.TELEGRAM_CHAT_ID",
    "resp = requests.post(",
    '    "https://api.telegram.org/bot" + token + "/sendMessage",',
    '    data={"chat_id": chat_id, "text": msg, "parse_mode": "HTML", "disable_web_page_preview": "true"},',
    "    timeout=15,",
    ")",
    'print("Conciertos enviados: status=" + str(resp.status_code) + ", " + str(len(picked)) + " eventos.")',
]

with open("concerts_cron.py", "w", encoding="utf-8") as f:
    f.write("\n".join(CONCERTS_PY_LINES) + "\n")

CRON_SH_LINES = [
    "#!/bin/bash",
    "# Lunes 9:00: enviar cartelera filtrada + conciertos favoritos por Telegram.",
    "# Configurar con `crontab -e`:",
    "#   0 9 * * 1 /home/anaya/Desktop-Ub/SSII/cron_weekly.sh",
    "set -e",
    'cd "$(dirname "$0")"',
    "",
    "# Cartelera de Madrid filtrada por perfil -> Telegram",
    "python3 cartelera_scraper.py --telegram",
    "",
    "# Conciertos de la semana filtrados por artistas favoritos -> Telegram",
    "python3 concerts_cron.py",
    "",
]
with open("cron_weekly.sh", "w", encoding="utf-8") as f:
    f.write("\n".join(CRON_SH_LINES))

os.chmod("cron_weekly.sh", 0o755)
os.chmod("concerts_cron.py", 0o755)

print("Archivos generados:")
print("  - cron_weekly.sh   (entry-point para cron)")
print("  - concerts_cron.py (logica de conciertos, ejecutable solo)")
print()
print("Para activar cada lunes a las 9:00:")
print("  crontab -e")
print("  0 9 * * 1 /home/anaya/Desktop-Ub/SSII/cron_weekly.sh")
print()
print("Probar manualmente:")
print("  ./cron_weekly.sh")


Archivos generados:
  - cron_weekly.sh   (entry-point para cron)
  - concerts_cron.py (logica de conciertos, ejecutable solo)

Para activar cada lunes a las 9:00:
  crontab -e
  0 9 * * 1 /home/anaya/Desktop-Ub/SSII/cron_weekly.sh

Probar manualmente:
  ./cron_weekly.sh


## 15. Demo final end-to-end

Una sola query natural en lenguaje libre dispara el flujo completo:
agente -> tools (scraping + LLM) -> formato -> Telegram + ICS.


In [21]:
# Demo 1: pelicula concreta
print(ask("cual es la nota de Interstellar?"))
print()


La nota de la película "Interstellar" es 4.4/5 con 4097 votos. La dirección fue realizada por Christopher Nolan y el género es Ciencia ficción, Drama. La duración de la película es de 2h 49min. La historia se centra en un grupo de exploradores que se adentran por uno de los agujeros de gusano para viajar en el tiempo.



In [22]:
# Demo 2: conciertos favoritos
print(ask("dame mis conciertos favoritos de esta semana"))
print()


Mis conciertos favoritos para esta semana en Madrid:

[2026-05-07 21:00] Concierto de Eric Clapton en Movistar Arena (Favorito)
[2026-05-08 20:30] Concierto de Fito & Fitipaldis en Movistar Arena (Favorito)
[2026-05-09 20:30] Concierto de Fito & Fitipaldis en Movistar Arena (Favorito)

¡No te pierdas nada!



In [23]:
# Demo 3: respuesta a correo
print(ask("responde a este mensaje de cliente: 'Llego tarde el pedido y la pizza estaba fria'"))


Por supuesto, puedo ayudarte con eso. ¿Podrías proporcionarme más detalles sobre tu experiencia? Por ejemplo, ¿cuál fue el nombre del restaurante donde te sentaste?


In [24]:
# Demo 4: generar calendario de conciertos
print(ask("exporta los conciertos al calendario"))


Los conciertos han sido exportados correctamente al calendario. Aquí está el archivo .ics:

```plaintext
[Archivo no visualizable aquí]
```

¡Disfruta de tus conciertos en Madrid!
